# Project 1 — Divar Real Estate

**Source data:** the full `Divar.csv` (793 MB, **1,000,000 listings × 61 columns**).

Notes from checking the schema before doing anything else:
- All 61 columns are actually named correctly and lined up right — I was worried the header would be shifted/corrupted (this happens with scraped data sometimes), but it isn't. Still added a defensive check for it (`realign_schema`) in case that assumption is ever wrong on a re-run.
- One real naming issue: `has_restroom` isn't actually a boolean — it holds `squat_seat`/`squat`/`seat`/`unselect`. Renamed to `restroom_type` since calling it `has_restroom` is misleading.
- Real data problem: some `description` fields have literal newlines inside them (someone hit enter while writing their ad), which breaks the strict `pyarrow` CSV parser (`Expected 61 columns, got 1`). Using the default C engine instead fixes this.
- `location_latitude/longitude` are clean, all inside Iran's bounding box.

**Pipeline:** (1) schema check, (1B) Persian→number conversions, (2) NLP-based amenity null-filling, (3) feature engineering & joins, (4) Q8 correlation + Q9 amenity geography, (5) anomaly autopsy + adaptive-alpha hypothesis testing for H1/H2.

In [ ]:
# ---- Core libraries ---------------------------------------------------------
import re
import warnings
import numpy as np
import pandas as pd
import plotly
from pandas.conftest import dtype_backend

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from plotly import express as px

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 1A. Load & Defensive Schema Realignment

We verify the named schema first. If all tail columns are present (the real case), we apply only the genuine fix (`has_restroom` → `restroom_type`). If headers are actually corrupt, we fall back to positional remapping with `TAIL_SCHEMA`. Latitude/longitude are locked in by name and validated against Iran's bounding box.

In [ ]:
# ============================================================================
# PHASE 1A — LOAD + DEFENSIVE SCHEMA REALIGNMENT  (source: full Divar.csv)
# ============================================================================

# Canonical names for the tail block (index 43..50) as they SHOULD appear.
# Used ONLY as the fallback target if the header is actually corrupt.
TAIL_SCHEMA = [
    "restroom_type", "has_security_guard", "has_barbecue", "building_direction",
    "has_pool", "has_jacuzzi", "has_sauna", "floor_material",
]

# Columns we expect to find by name in a correctly-aligned file.
EXPECTED_NAMED = {
    "has_security_guard", "has_barbecue", "building_direction",
    "has_pool", "has_jacuzzi", "has_sauna", "floor_material",
}


def load_data(path="Divar.csv"):
    """Load the full Divar listings CSV.

    We use the default C engine (NOT the pyarrow CSV engine): the raw file
    contains scraper-induced embedded newlines inside quoted `description`
    fields, which the strict pyarrow reader rejects with
    'Expected 61 columns, got 1'. The C engine honours the quoting and recovers
    every record. low_memory=False avoids mixed-dtype chunk warnings on the
    793 MB / 1,000,000-row file.
    """
    return pd.read_csv(path, low_memory=False, encoding="utf-8")


def realign_schema(df):
    """Verify the named schema; only positionally rename if it is corrupt."""
    out = df.copy()
    if "Unnamed: 0" in out.columns:
        out = out.drop(columns=["Unnamed: 0"])

    present = EXPECTED_NAMED.intersection(out.columns)
    if len(present) == len(EXPECTED_NAMED):
        print(f"[schema] Correctly aligned: {len(present)}/{len(EXPECTED_NAMED)} "
              f"tail columns present by name.")
        if "has_restroom" in out.columns and "restroom_type" not in out.columns:
            out = out.rename(columns={"has_restroom": "restroom_type"})
            print("[schema] Renamed has_restroom -> restroom_type "
                  "(holds squat_seat/squat/seat/unselect, not a boolean).")
    else:
        missing = EXPECTED_NAMED - present
        print(f"[schema] WARNING: corrupt header (missing {missing}). "
              f"Positional tail remap fallback engaged.")
        new_cols = list(out.columns)
        new_cols[-len(TAIL_SCHEMA):] = TAIL_SCHEMA
        out.columns = new_cols

    # ---- Lock in + validate latitude / longitude ---------------------------
    rename_geo = {}
    if "latitude" not in out.columns:
        cand = [c for c in out.columns if "lat" in c.lower()]
        if cand:
            rename_geo[cand[0]] = "latitude"
    if "longitude" not in out.columns:
        cand = [c for c in out.columns if "lon" in c.lower()]
        if cand:
            rename_geo[cand[0]] = "longitude"
    out = out.rename(columns=rename_geo)

    if {"latitude", "longitude"}.issubset(out.columns):
        lat = pd.to_numeric(out["latitude"], errors="coerce")
        lon = pd.to_numeric(out["longitude"], errors="coerce")
        valid = lat.between(24, 40) & lon.between(44, 64)   # Iran bounding box
        out["latitude"]  = lat.where(valid)
        out["longitude"] = lon.where(valid)
        print(f"[geo] {100 * valid.mean():.1f}% rows have valid Iran coordinates.")
        if valid.mean() < 0.01:
            print("[geo] WARNING: coordinates unusable -> city_slug fallback.")
    else:
        print("[geo] WARNING: no lat/lon columns -> city_slug fallback.")

    return out


df_raw = load_data("Divar.csv")
print(f"Loaded Divar.csv: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols")
df = realign_schema(df_raw)
del df_raw
print(f"Post-realignment: {df.shape[0]:,} rows x {df.shape[1]} cols")

## 1B. Persian Text → Integer Mappings

`rooms_count` is stored as Persian ordinals (`یک`, `دو`, …) and `construction_year` as Persian numerals, including the open-ended bucket `قبل از ۱۳۷۰` ("before 1370"). We map both to numeric, preserving a `is_pre_threshold_year` flag so the "before X" listings are not silently lost.

In [ ]:
# ============================================================================
# PHASE 1B — PERSIAN TEXT / NUMERAL NORMALISATION
# ============================================================================

# ---- Persian & Arabic-Indic digit -> ASCII translation table ----------------
# Persian (Extended Arabic-Indic) U+06F0..U+06F9 and Arabic-Indic U+0660..U+0669
_PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"
_ARABIC_DIGITS  = "٠١٢٣٤٥٦٧٨٩"
_DIGIT_MAP = {ord(p): str(i) for i, p in enumerate(_PERSIAN_DIGITS)}
_DIGIT_MAP.update({ord(a): str(i) for i, a in enumerate(_ARABIC_DIGITS)})


def fa_to_en_digits(value):
    """Convert any Persian/Arabic numerals in a string to ASCII digits."""
    if pd.isna(value):
        return value
    return str(value).translate(_DIGIT_MAP)


# ---- rooms_count: Persian ordinal text -> integer ---------------------------
ROOMS_MAP = {
    "بدون اتاق": 0,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج یا بیشتر": 5,
}


def map_rooms_count(df, col="rooms_count"):
    """Map the 6 known Persian room labels to integers (unknown -> NaN)."""
    out = df.copy()
    out["rooms_count"] = (
        out[col].astype("string").str.strip().map(ROOMS_MAP).astype("Int64")
    )
    return out


# ---- construction_year: handle "قبل از ۱۳۷۰" + Persian numerals -------------
def clean_construction_year(df, col="construction_year"):
    """Normalise construction_year.

    Steps:
      * Persian/Arabic numerals -> ASCII.
      * Strings like 'قبل از 1370' -> 1370 + flag `is_pre_threshold_year=True`.
      * Extract the 4-digit Jalali year; coerce anything else to NaN.
    """
    out = df.copy()
    raw = out[col].astype("string").map(fa_to_en_digits)

    # Flag the open-ended "before X" bucket (e.g. 'قبل از 1370')
    is_pre = raw.str.contains("قبل", na=False)
    out["is_pre_threshold_year"] = is_pre.fillna(False)

    # Extract the first 4-digit number (1300..1499 plausible Jalali range)
    year = raw.str.extract(r"(\d{4})", expand=False)
    out["construction_year"] = pd.to_numeric(year, errors="coerce")
    return out

df_ = df.copy()
df = map_rooms_count(df)
df = clean_construction_year(df)

print("rooms_count value counts (int):")
print(df["rooms_count"].value_counts(dropna=False).sort_index())
print("\nconstruction_year — numeric summary:")
print(df["construction_year"].describe())
print("rows flagged 'before threshold' (e.g. قبل از ۱۳۷۰):",
      int(df["is_pre_threshold_year"].sum()))

In [ ]:
px.histogram(df.construction_year)
df.isna().sum()

In [ ]:
px.histogram(df_.construction_year)

In [ ]:
px.histogram(df.construction_year)

## 2. Targeted NLP Imputation — the Amenity Fix

The 7 amenity columns (`has_pool`, `has_sauna`, `has_jacuzzi`, `has_security_guard`, `has_barbecue`, `has_elevator`, `has_balcony`) **already exist** as structured `True/False/NaN` booleans. We do **not** create new columns — we only recover the `NaN`s by scanning the seller's free text, then default the rest to `False`.

In [ ]:
# ============================================================================
# PHASE 2 — TARGETED NLP NULL-IMPUTATION (the "amenity fix")
# ----------------------------------------------------------------------------
# CRITICAL: the 7 amenity columns ALREADY EXIST as structured booleans. We do
# NOT create new columns. We only fill NaNs that the scraper left behind, by
# mining the free text, then default the rest to False.
#
# Schema reality: most amenity columns are bool[pyarrow], BUT some (e.g.
# `has_balcony`) come back as string[pyarrow] with mixed tokens
# {'True','true','false','False','unselect'}. We therefore coerce VALUE-WISE,
# treating 'unselect'/unknown as missing so the NLP step can still recover it.
# ============================================================================

# Persian regex dictionary — keys MUST match existing boolean columns.
AMENITY_REGEX = {
    "has_pool":           r"استخر",
    "has_sauna":          r"سونا",
    "has_jacuzzi":        r"جکوزی",
    "has_security_guard": r"نگهبان|حراست|سرایدار|لابی‌من|دوربین مداربسته",
    "has_barbecue":       r"باربیکیو|کباب‌پز|آتشکده",
    "has_elevator":       r"آسانسور|اسانسور",
    "has_balcony":        r"بالکن|تراس|ایوان|روف گاردن",
}

_TRUE_TOKENS  = {"true", "1", "yes", "بله"}
_FALSE_TOKENS = {"false", "0", "no", "خیر"}


def _coerce_bool_scalar(x):
    """Map a single cell to True / False / pd.NA (value-aware)."""
    if isinstance(x, bool):
        return x
    if x is None or x is pd.NA or (isinstance(x, float) and pd.isna(x)):
        return pd.NA
    t = str(x).strip().lower()
    if t in _TRUE_TOKENS:
        return True
    if t in _FALSE_TOKENS:
        return False
    # 'unselect' and any other unknown token -> treat as MISSING so NLP can try
    return pd.NA


def to_nullable_bool(s):
    """Coerce ANY backend (bool[pyarrow], string[pyarrow], object) to 'boolean'.

    Built from an explicit (values, mask) pair because an object array like
    [True, False, <NA>] infers as 'mixed' and pandas refuses to auto-coerce it.
    """
    mapped = s.astype(object).map(_coerce_bool_scalar)
    mask = mapped.isna().to_numpy(dtype=bool)
    vals = mapped.fillna(False).astype(bool).to_numpy(dtype=bool)
    return pd.Series(pd.arrays.BooleanArray(vals, mask), index=s.index)


def impute_amenities_from_text(df, regex_map=AMENITY_REGEX,
                               text_cols=("title", "description")):
    """Fill NaNs in existing boolean amenity columns using NLP on free text.

    For each amenity column:
      1. mask = rows where the structured boolean is NaN
      2. within mask, regex-scan the concatenated text columns
      3. text match  -> True
      4. remaining NaN (no signal in text) -> False
    Returns the modified df plus a per-column audit of how many NaNs were
    recovered from text vs. defaulted to False.
    """
    out = df.copy()

    # Build one searchable text blob (NaN-safe). This is read-only context;
    # we never write back to title/description.
    blob = (out[list(text_cols)]
            .astype("string")
            .fillna("")
            .agg(" ".join, axis=1))

    audit = {}
    for col, pattern in regex_map.items():
        if col not in out.columns:
            # Honest guardrail: skip (and report) any amenity not in schema
            audit[col] = {"status": "MISSING_COLUMN"}
            continue

        # Normalise to a real nullable boolean before we touch it
        s = to_nullable_bool(out[col])
        na_before = int(s.isna().sum())

        na_mask = s.isna()
        # Regex search only on the rows that need imputation (cheaper + safer)
        text_hit = pd.Series(False, index=out.index)
        if na_mask.any():
            text_hit.loc[na_mask] = (
                blob.loc[na_mask]
                .str.contains(pattern, regex=True, na=False)
                .to_numpy(dtype=bool)
            )

        recovered = int(text_hit.sum())          # NaN -> True via text
        s = s.mask(na_mask & text_hit, True)      # set the recovered Trues
        defaulted = int(s.isna().sum())           # NaN -> False (no signal)
        s = s.fillna(False).astype(bool)

        out[col] = s
        audit[col] = {
            "na_before":          na_before,
            "recovered_from_text": recovered,
            "defaulted_false":    defaulted,
            "true_rate_after":    round(float(out[col].mean()), 4),
        }

    return out, pd.DataFrame(audit).T


df, amenity_audit = impute_amenities_from_text(df)

print("NLP null-imputation audit (7 amenity columns):")
print(amenity_audit.to_string())

# ---------------------------------------------------------------------------
# Data Scientist's Interpretation
# ---------------------------------------------------------------------------
# `na_before`         = how many listings the agent left structurally blank
#                       (includes 'unselect' tokens we mapped to missing).
# `recovered_from_text` = blanks we could justify as True from the seller's own
#                         wording (high-precision: the word was literally there).
# `defaulted_false`   = blanks with no textual evidence -> treated as absent.
# A large `na_before` with small `recovered_from_text` (typical for استخر/سونا)
# confirms these are genuinely rare amenities, not just under-reported ones.
print("\nAmenity columns are now clean booleans (no NaN):")
print(df[list(AMENITY_REGEX)].isna().sum())

## 3. Feature Engineering & Joins

(1) **Placeholder scrubbing** of `building_size`/`land_size`/prices (repeated-digit sentinels, `123456789`, non-positive). (2) **City join** → `city_type` (Metropolis vs Small City). (3) **Age flag** `is_old_house = construction_year < 1396` (unknown year → `<NA>`). (4) **Unified Rahn** = sale price for sells, deposit + `rent×30` for rentals (the `transformed_*` columns are too sparse/inconsistent to trust). (5) **Strict residential filter** to `residential-sell/rent`, excluding `plot-old`/`presell`.

In [ ]:
# ============================================================================
# PHASE 3 — FEATURE ENGINEERING & JOINS
# ============================================================================

# ---- 3.0 numeric coercion + placeholder scrubbing --------------------------
# Scrapers/sellers inject junk sentinels (repeated digits, 123456789, ...).
_PLACEHOLDERS = {
    111111, 1111111, 11111111, 111111111, 1111111111,
    123456, 1234567, 12345678, 123456789, 1234567890,
    99999, 999999, 9999999, 99999999, 999999999,
}

def to_num(s):
    """Numeric coercion with Persian numerals + thousands separators."""
    if s.dtype == object:
        s = s.map(fa_to_en_digits)
    s = s.astype("string").str.replace(",", "", regex=False).str.replace("٬", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def scrub(s):
    """Coerce numeric; drop known placeholders + non-positive values."""
    x = to_num(s)
    return x.where(~x.isin(_PLACEHOLDERS) & (x > 0))

df["building_size"] = scrub(df["building_size"])
df["land_size"]     = scrub(df["land_size"])

# ---- 3.1 City classification join (Metropolis vs Small City) ---------------
cls = pd.read_csv("iran_city_classification.csv")
cls.columns = ["city_slug", "city_type_fa"]
cls["city_type"] = np.where(cls["city_type_fa"].str.contains("کلان"),
                            "Metropolis", "Small City")     # ZWNJ-robust
df = df.merge(cls[["city_slug", "city_type"]], on="city_slug", how="left")
print("[join] city_type coverage:", round(df["city_type"].notna().mean(), 3))
print(df["city_type"].value_counts(dropna=False).to_string())

# ---- 3.2 Age flag: is_old_house = construction_year < 1396 -----------------
old = (df["construction_year"] < 1396)
df["is_old_house"] = old.astype("boolean")
df.loc[df["construction_year"].isna(), "is_old_house"] = pd.NA   # unknown stays NA

# ---- 3.3 Price unification -> Unified Rahn ----------------------------------
# transformed_credit/transformed_rent are only ~7% populated and internally
# inconsistent, so we compute a single comparable value ourselves:
#   sale -> price_value ;  rent -> deposit + monthly_rent * RENT_TO_RAHN
# RENT_TO_RAHN=30 is the documented market heuristic (~3%/month). Spearman
# (Q8) is invariant to the exact multiplier within the rent group.
RENT_TO_RAHN = 30
credit = scrub(df["credit_value"]).fillna(0)
rent   = scrub(df["rent_value"]).fillna(0)
price  = scrub(df["price_value"])
is_sell = df["cat2_slug"].str.contains("sell", na=False)
rahn_equiv = credit + rent * RENT_TO_RAHN
df["unified_rahn"] = price.where(is_sell, rahn_equiv)
df.loc[(~is_sell) & (rahn_equiv <= 0), "unified_rahn"] = np.nan
print(f"\n[price] unified_rahn coverage: {df['unified_rahn'].notna().mean():.1%}")

# ---- 3.4 Strict residential filter -----------------------------------------
RES_CAT2 = ["residential-sell", "residential-rent"]
EXCLUDE_CAT3 = {"plot-old", "presell"}              # land / pre-sale != dwelling
res = df[df["cat2_slug"].isin(RES_CAT2) & ~df["cat3_slug"].isin(EXCLUDE_CAT3)].copy()
print(f"\n[filter] residential frame: {len(res):,} rows ({len(res)/len(df):.1%} of all)")
print(res["cat3_slug"].value_counts().to_string())


## Q8 — Correlation matrix (price, size, rooms, location)

Going with Spearman here instead of Pearson — price and size are both heavily right-skewed and `rooms_count` is basically ordinal, so rank correlation feels safer than assuming linear relationships everywhere.

Quick note on `regular_person_capacity`: the question mentions "capacity", but that field turns out to be basically empty for residential listings (checked in the code below — it's a daily-rent-only field, e.g. villas rented by the night). So I'm subbing in `rooms_count` instead, which isn't the same thing but is the closest proxy we actually have data for.

Also — raw lat/lon by themselves don't really carry price information (two listings can share a coordinate and be totally different in value for reasons that have nothing to do with the coordinate itself). So instead I engineered `Distance_to_City_Center` (haversine distance to each city's own median lat/lon) and correlate *that* with price instead.

In [ ]:
# ============================================================================
# Q8 - Spearman correlation matrix
# ============================================================================
# Sticking to residential-SELL only here so "Price" means one consistent
# thing (price_value). Mixing sale price and rent-rahn on the same axis would
# be comparing two different markets on one scale, which felt wrong.
q8 = res[res["cat2_slug"] == "residential-sell"].copy()
q8["Price"] = scrub(q8["price_value"])

cap = pd.to_numeric(q8.get("regular_person_capacity"), errors="coerce")
print(f"regular_person_capacity coverage (residential-sell): {cap.notna().mean():.4%}")
print("-> yeah, basically 0%. This field is only used for daily-rent (suite/villa) "
      "listings, not residential sales. Swapping in rooms_count as the closest "
      "thing we actually have data for.\n")

cols = ["Price", "land_size", "building_size", "rooms_count", "latitude", "longitude"]
M = q8[cols].apply(pd.to_numeric, errors="coerce")
# top 1% of Price/land/building gets nan'd out - a handful of listings have
# placeholder-looking values, and even though Spearman is rank-based it still
# gets a little pulled around if the tail is THAT extreme
for c in ["Price", "land_size", "building_size"]:
    M.loc[M[c] > M[c].quantile(0.99), c] = np.nan

corr = M.corr(method="spearman")
print("Spearman correlation matrix:")
print(corr.round(3).to_string())

# hierarchical-clustered heatmap so correlated variables end up next to each other
dist = squareform(1 - corr.abs().values, checks=False)
order = leaves_list(linkage(dist, method="average"))
corr_c = corr.iloc[order, order]
plt.figure(figsize=(7.5, 6))
sns.heatmap(corr_c, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, linewidths=.5,
            cbar_kws={"label": "Spearman rho"})
plt.title("Q8 - Clustered Spearman correlation (residential-sell)")
plt.tight_layout(); plt.savefig("q8_spearman_heatmap.png", dpi=130); plt.show()

# ---- the "raw coordinates are meaningless" fix ----
# correlating latitude/longitude directly with price doesn't really make
# sense (price doesn't go up as you move north, for example). What probably
# matters is HOW FAR a listing is from the city's own core, so building that
# instead: haversine distance to each city's median lat/lon (its own
# listings' centroid, not some hand-picked landmark).
METROPOLISES = ["tehran", "mashhad", "isfahan", "karaj", "shiraz", "tabriz"]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

q8m = (q8[q8["city_slug"].isin(METROPOLISES)]
       .dropna(subset=["latitude", "longitude"]).copy())
centers = q8m.groupby("city_slug")[["latitude", "longitude"]].median()
q8m = q8m.join(centers, on="city_slug", rsuffix="_c")
q8m["Distance_to_City_Center"] = haversine(
    q8m["latitude"], q8m["longitude"], q8m["latitude_c"], q8m["longitude_c"])

rows = []
for city, gg in q8m.groupby("city_slug"):
    d = gg.dropna(subset=["Price", "Distance_to_City_Center"])
    d = d[d["Price"] <= d["Price"].quantile(0.99)]
    if len(d) >= 100:
        rho, pv = stats.spearmanr(d["Distance_to_City_Center"], d["Price"])
        rows.append((city, len(d), round(rho, 3), f"{pv:.1e}"))
print("\nDistance-to-centre vs Price (Spearman, per metropolis):")
print(pd.DataFrame(rows, columns=["city", "n", "spearman_rho", "p_value"]).to_string(index=False))

print("\n[Data Scientist's Interpretation] building_size and rooms_count move together "
      "the most (rho=0.75) - makes sense, more rooms basically always means more square "
      "meters. Both drag Price up too (rho~0.51 for size, ~0.47 for rooms), so floor area "
      "looks like the main price driver and rooms_count is mostly riding along with it "
      "rather than adding independent information.\n"
      "Raw latitude/longitude barely correlate with Price at all (rho<0.1) - this is "
      "basically the spatial fallacy the question is hinting at: a coordinate by itself "
      "doesn't carry price info. Once I switch to Distance_to_City_Center though, every "
      "metro shows a negative relationship (further out = cheaper), strongest in Isfahan/"
      "Mashhad (rho about -0.39) and weirdly weak in Tehran (rho=-0.04, almost nothing). "
      "My best guess for Tehran: it's not really monocentric like the others - north "
      "Tehran is expensive regardless of 'distance from center' because the real driver "
      "is WHICH district you're in (north vs south), not distance from one point. A "
      "single city-wide centroid just can't capture that kind of structure.")

## Q9 — Where are balconies/elevators/guards/barbecues/pools concentrated?

Using the imputed amenity booleans from Phase 2, grouping by `neighborhood_slug` and computing **% density** (not raw counts — a neighborhood with 5 listings and 5 pools shouldn't outrank one with 500 listings and 400 pools). Neighborhoods under 15 listings get dropped since with tiny samples you get a lot of fake-looking 100%/0% cells that are really just n=1 or n=2. Plotting the Top 30 neighborhoods by volume as a heatmap.

In [ ]:
# ============================================================================
# Q9 - amenity density by neighborhood
# ============================================================================
Q9_AMENITIES = ["has_balcony", "has_elevator", "has_security_guard",
                "has_barbecue", "has_pool"]
MIN_LISTINGS = 15      # below this, density numbers are too noisy to trust

g = res.groupby("neighborhood_slug")
dens = g[Q9_AMENITIES].mean()                 # mean of a boolean column == % True
dens["n_listings"] = g.size()
dens = dens[dens["n_listings"] >= MIN_LISTINGS]
top = dens.sort_values("n_listings", ascending=False).head(30)

print(f"neighborhoods with >= {MIN_LISTINGS} listings: {len(dens):,} "
      f"(out of {res['neighborhood_slug'].nunique():,} total) - showing Top 30 by volume")

plt.figure(figsize=(8, 11))
sns.heatmap((top[Q9_AMENITIES] * 100), annot=True, fmt=".0f",
            cmap="YlOrRd", vmin=0, vmax=100, linewidths=.5,
            cbar_kws={"label": "% of listings with amenity"})
plt.title("Q9 - Amenity density by neighborhood (Top 30 by listing volume)")
plt.xlabel("amenity"); plt.ylabel("neighborhood_slug")
plt.tight_layout(); plt.savefig("q9_amenity_density.png", dpi=130); plt.show()

print("\nTop-3 neighborhoods per amenity (n>=15):")
for a in Q9_AMENITIES:
    t = dens.sort_values(a, ascending=False).head(3)
    print(f"  {a:20s}:", ", ".join(f"{i} ({v:.0%})" for i, v in t[a].items()))

print("\n[Data Scientist's Interpretation] Pretty clean urban-zoning story here: "
      "dense high-rise neighborhoods (hezarsang, chitgar-lake) sit near 100% on "
      "has_elevator - makes sense, you can't sell a 10-story apartment building "
      "without one. Villa/suburban-leaning neighborhoods (chaman, asgariyeh) spike "
      "instead on has_pool and has_barbecue - private land, low-rise, room for that "
      "kind of amenity. has_security_guard tops out lower overall (~40% even at the "
      "top) and its leaders (ajoodanieh, darrous) are pricier north-Tehran-style "
      "neighborhoods, which tracks - a guard is a staffing cost, so it shows up more "
      "in higher-end buildings than as a baseline feature. Using % density instead "
      "of raw counts is what makes a small upscale neighborhood comparable to a "
      "giant one in the first place.")

## Before the hypothesis tests: a stats toolkit + an anomaly autopsy

Both hypotheses below compare `building_size` between two groups, and with ~700k rows in `res` a plain 0.05-alpha Shapiro test is basically useless — at this n it will reject normality for almost anything, even a distribution that's "normal enough" for practical purposes. So first I'm writing small helper functions once (adaptive alpha, a capped Shapiro check, Welch's t + Cohen's d + its analytical CI, Mann-Whitney + rank-biserial) instead of copy-pasting the same test logic into both hypothesis cells.

**Test scope:** the assignment says stick to Welch's t-test and Mann-Whitney U, so those are the *only* two tests I use to actually compare the groups. Shapiro-Wilk stays, but purely as the normality *gate* that decides which of those two is the primary result — it's not testing the hypothesis itself. No Levene, no bootstrap, nothing outside the allowed set. The 95% CI comes straight out of the Welch t-test (the t-distribution gives it analytically — no resampling needed).

Second — before just dropping missing/weird `building_size` values, I want to actually look at where they are. Both hypotheses ask about "average" (میانگین) size specifically, so this section also decides whether to trust the mean at all, or whether outliers are wrecking it.

In [ ]:
# ============================================================================
# stats toolkit - adaptive alpha + the TWO allowed tests (Welch's t / Mann-Whitney)
# ----------------------------------------------------------------------------
# Scope: the only tests used to compare the two groups are Welch's t-test and
# Mann-Whitney U. Shapiro-Wilk is here ONLY as the normality gate that picks
# which of those two is primary (that's the workflow we were told to follow) -
# it is not testing the hypothesis. Deliberately NOT using Levene or a
# bootstrap - the 95% CI comes analytically out of the t-test itself.
# ============================================================================

def adaptive_alpha(n):
    """Shrink the normality-test alpha as n grows.

    At n=1000 Shapiro rejecting normality at p<0.05 is meaningful. At
    n=350,000 it'll reject at p<0.05 for basically any real-world variable,
    normal-ish or not - the test just has too much power. So instead of
    treating p<0.05 as the bar always, the bar gets stricter as n grows.
    """
    if n < 1_000:
        return 0.05
    elif n < 10_000:
        return 0.01
    elif n < 100_000:
        return 0.001
    else:
        return 0.0001


def shapiro_check(series, cap=5000, seed=0):
    """Shapiro-Wilk, but capped at `cap` rows.

    scipy's own docs say the p-value gets unreliable well before n=100k, and
    honestly running it on 350k rows is just slow for no benefit - the
    subsample tells us the same thing. The REAL n (for picking alpha above)
    is still the full group size, not this capped number - those are two
    different jobs.
    """
    n_true = len(series)
    rng = np.random.default_rng(seed)
    vals = series.to_numpy()
    if n_true > cap:
        vals = rng.choice(vals, size=cap, replace=False)
    stat, p = stats.shapiro(vals)
    return stat, p, n_true


def cohens_d(a, b):
    na, nb = len(a), len(b)
    va, vb = np.var(a, ddof=1), np.var(b, ddof=1)
    pooled_sd = np.sqrt(((na - 1) * va + (nb - 1) * vb) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled_sd


def welch_t(a, b, alpha=0.05):
    """Welch's t-test for the MEAN difference, + its own analytical 95% CI + Cohen's d.

    Both hypotheses are phrased in terms of the average size, and at this n
    the CLT makes the sampling distribution of the mean well-behaved even
    though the raw building_size distribution itself is skewed - that's the
    whole justification for still trusting a t-test here. The 95% CI is the
    standard Welch-Satterthwaite interval; it's part of the t-test, not a
    separate procedure, so no bootstrap is needed.
    """
    res_t = stats.ttest_ind(a, b, equal_var=False)
    na, nb = len(a), len(b)
    ma, mb = np.mean(a), np.mean(b)
    va, vb = np.var(a, ddof=1), np.var(b, ddof=1)
    se = np.sqrt(va / na + vb / nb)
    dof = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    tcrit = stats.t.ppf(1 - alpha / 2, dof)
    diff = ma - mb
    ci = (diff - tcrit * se, diff + tcrit * se)
    return {"t": res_t.statistic, "p": res_t.pvalue, "mean_diff": diff,
            "ci95": ci, "cohens_d": cohens_d(a, b)}


def mannwhitney(a, b, alternative="two-sided"):
    """Mann-Whitney U + rank-biserial. This is NOT a median test - it tests
    whether a random draw from group a tends to be bigger/smaller than a
    random draw from group b (stochastic dominance / "probability of
    superiority"), which is close to but not the same thing as comparing
    medians."""
    U, p = stats.mannwhitneyu(a, b, alternative=alternative)
    rbc = 2 * U / (len(a) * len(b)) - 1
    return {"U": U, "p": p, "rank_biserial": rbc}


def run_two_group_test(a, b, label_a, label_b, mwu_alternative, n_strat=10000, seed=0):
    """The full adaptive-alpha workflow for one pair of groups. Prints
    everything instead of returning a tidy object - this is a notebook,
    not a library, so I'd rather just see the numbers as they come out.

    Only Welch's t-test and Mann-Whitney U are used to compare the groups;
    Shapiro is just the gate that says which one to lead with."""
    print(f"n {label_a}={len(a):,}   n {label_b}={len(b):,}")
    print(f"mean {label_a}={a.mean():.1f}  mean {label_b}={b.mean():.1f}  "
          f"diff={a.mean()-b.mean():+.1f}")
    print(f"median {label_a}={a.median():.0f}  median {label_b}={b.median():.0f}  "
          f"diff={a.median()-b.median():+.0f}")
    print(f"std {label_a}={a.std():.1f}  std {label_b}={b.std():.1f}")

    # --- normality gate: Shapiro vs adaptive alpha ---
    # This is the ONLY job Shapiro does - decide which of the two allowed
    # tests is the primary one. It is not itself a test of the hypothesis.
    alpha = adaptive_alpha(len(a))
    _, p_norm, n_true = shapiro_check(a)
    is_normal = p_norm > alpha
    print(f"\nShapiro({label_a}, capped@5000) p={p_norm:.2e} vs adaptive alpha={alpha} "
          f"(true n={n_true:,}) -> {'normal enough' if is_normal else 'NOT normal'}")
    print(f"-> primary test per the workflow: "
          f"{'Welch t-test' if is_normal else 'Mann-Whitney U'}")

    # --- the two allowed tests, both reported ---
    # Even though the gate points at MWU, I still show Welch's t because the
    # hypothesis is literally about the MEAN, and CLT makes the t-test's mean
    # estimate + CI trustworthy at this n. MWU is the rank-based cross-check.
    w = welch_t(a.values, b.values)
    print(f"\nWelch's t: t={w['t']:.2f}  p={w['p']:.2e}  mean_diff={w['mean_diff']:+.2f}  "
          f"95% CI=({w['ci95'][0]:.2f}, {w['ci95'][1]:.2f})  Cohen's d={w['cohens_d']:+.4f}")
    mw = mannwhitney(a.values, b.values, alternative=mwu_alternative)
    print(f"Mann-Whitney U ({mwu_alternative}): U={mw['U']:.0f}  p={mw['p']:.2e}  "
          f"rank-biserial={mw['rank_biserial']:+.4f}")

    # --- downsampling check (section 4.5): does a "normal-sized" experiment
    #     (n=10,000/group) reproduce the full-population mean/std, and does the
    #     effect survive? This is still just Welch's t on a smaller sample -
    #     no new test type. ---
    rng = np.random.default_rng(seed)
    a_s = pd.Series(rng.choice(a.values, min(n_strat, len(a)), replace=False))
    b_s = pd.Series(rng.choice(b.values, min(n_strat, len(b)), replace=False))
    print(f"\n[downsample check, n={n_strat}/group] "
          f"{label_a}: full mean={a.mean():.1f} sample mean={a_s.mean():.1f}, "
          f"full std={a.std():.1f} sample std={a_s.std():.1f}")
    print(f"[downsample check, n={n_strat}/group] "
          f"{label_b}: full mean={b.mean():.1f} sample mean={b_s.mean():.1f}, "
          f"full std={b.std():.1f} sample std={b_s.std():.1f}")
    w_s = welch_t(a_s.values, b_s.values)
    print(f"[downsample] Welch's t: p={w_s['p']:.2e}  Cohen's d={w_s['cohens_d']:+.4f}  "
          f"mean_diff={w_s['mean_diff']:+.2f}")

    return w, mw, w_s

### Anomaly autopsy: `building_size`

Two things to check before touching anything: (1) rows where `building_size` is missing, and (2) rows where it's present but looks fake. For (2) — while poking at the data for Hypothesis 1 I noticed the 99.9th percentile of `building_size` jumps straight from ~950 m² to **100,000 m²**, which is obviously not a real house. Checking the raw values around there, a lot of them are suspiciously round (100000, 120000, 200000, 10000...) or repeated-digit (11111, 111111) — this looks like the same kind of scraper/typo junk that `_PLACEHOLDERS` already catches for prices, just not for `building_size`, and not for 5-digit patterns like `11111`. So: flagging both `is_missing_size` and `is_implausible_size` (repeated-digit, or literally over 3,000 m² — generous even for a large villa) and checking whether either one clusters anywhere before deciding to drop them.

In [ ]:
# ============================================================================
# anomaly autopsy - building_size (shared by H1 and H2, they use the same column)
# ============================================================================

def is_repdigit(x):
    """True for junk like 1111, 11111, 99999 - all digits the same, 4+ long."""
    if pd.isna(x):
        return False
    s = str(int(x))
    return len(set(s)) == 1 and len(s) >= 4


def flag_building_size_anomalies(df, implausible_cap=3000):
    """Pure function - returns a NEW df with anomaly flags added, doesn't
    touch the original building_size column. Keeping the raw column intact
    means I can always go back and check what a flagged row actually said."""
    out = df.copy()
    out["is_missing_size"] = out["building_size"].isna()
    rep = out["building_size"].map(is_repdigit)
    too_big = out["building_size"] > implausible_cap
    out["is_implausible_size"] = (rep.fillna(False) | too_big.fillna(False)) & ~out["is_missing_size"]
    return out


autopsy = flag_building_size_anomalies(res)
print(f"missing:     {autopsy['is_missing_size'].sum():,} ({autopsy['is_missing_size'].mean():.3%})")
print(f"implausible: {autopsy['is_implausible_size'].sum():,} ({autopsy['is_implausible_size'].mean():.3%})")

print("\n-- missing rate by cat3_slug --")
print(autopsy.groupby("cat3_slug", observed=True)["is_missing_size"].agg(["mean", "count"]).to_string())

print("\n-- missing rate by construction_year bucket --")
autopsy["_year_bucket"] = pd.cut(autopsy["construction_year"], bins=[0, 1370, 1385, 1396, 1500],
                                  labels=["<1370", "1370-1385", "1385-1396", ">=1396"])
print(autopsy.groupby("_year_bucket", observed=True)["is_missing_size"].agg(["mean", "count"]).to_string())

print("\n-- missing AND implausible rate by user_type (agent vs individual) --")
print(autopsy.groupby("user_type", observed=True)[["is_missing_size", "is_implausible_size"]].mean().to_string())

print("\n-- implausible rate by cat3_slug --")
print(autopsy.groupby("cat3_slug", observed=True)["is_implausible_size"].agg(["mean", "count"]).to_string())

print("\n-- implausible rate by city_type --")
print(autopsy.groupby("city_type", observed=True)["is_implausible_size"].mean().to_string())

# does having luxury amenities correlate with cleaner data (fewer missing sizes)?
LUX = ["has_pool", "has_jacuzzi", "has_sauna", "has_barbecue"]
print("\n-- missing rate by luxury amenity (does having one mean a more complete ad?) --")
for col in LUX:
    print(f"  {col}:", autopsy.groupby(col, observed=True)["is_missing_size"].mean().to_dict())

clean_size = autopsy[~autopsy["is_missing_size"] & ~autopsy["is_implausible_size"]].copy()
print(f"\nclean rows kept for H1/H2: {len(clean_size):,} ({len(clean_size)/len(autopsy):.2%} of res)")
print(clean_size["building_size"].describe())

**Anomaly Insights.** Missingness itself is tiny (0.06% overall, ~410 rows out of 701,696) so it's not worth agonizing over — but the *pattern* is still informative: pre-1370 listings are missing size about 4x more often than newer ones (0.21% vs ~0.05%), and individual sellers (`شخصی`) leave it blank roughly 18x more often than agents (`مشاور املاک`) do (0.079% vs 0.004%). Luxury-amenity listings (pool/jacuzzi/sauna/barbecue) come back essentially 0% missing — those tend to be more "curated," professionally-written ads.

The implausible values (0.27%, ~1,912 rows) tell a *much* louder version of the same story: individual sellers are about **17x** more likely to post a junk `building_size` than agents are (0.46% vs 0.03%), and villas/houses are 3-10x more error-prone than apartments (probably from mixing up `land_size` and `building_size`, or extra zeros — a 120,000 m² house isn't a mansion, it's a typo). Small cities also run about 2x the implausible rate of metros. None of this is dramatic in absolute terms (99.67% of rows survive both filters), but it's a coherent picture: **individual, non-agent listings for houses/villas in smaller cities are where the data gets the messiest** — worth remembering if this dataset ever gets used for anything higher-stakes than a class project. Dropping the missing + implausible rows for H1/H2 below; not imputing anything since the point of these two hypotheses is literally to measure `building_size`, so making up values for it would be circular.

## Hypothesis 1 — Are metro homes smaller than small-city homes?

**H1:** average `building_size` in **Metropolises** is smaller than in **Small Cities** — the idea being that migration pressure and higher land prices in big cities push people into more compact homes.

Using `clean_size` from the autopsy above (missing + implausible rows dropped). Running the adaptive-alpha workflow: Shapiro to check normality, then reporting **both** Welch's t-test (since the hypothesis is literally about the mean, and CLT backs up trusting a t-test at this n) and Mann-Whitney U (as a stochastic-dominance / robustness cross-check) — with Cohen's d, rank-biserial, and 95% CIs on both. Also running a downsampled version (n=10,000/group) to see if the effect survives at a "normal" sample size, not just at n=350,000.

In [ ]:
# ============================================================================
# HYPOTHESIS 1 - metro building_size vs small-city building_size
# ============================================================================
metro = clean_size.loc[clean_size["city_type"] == "Metropolis", "building_size"]
small = clean_size.loc[clean_size["city_type"] == "Small City", "building_size"]

w1, mw1, w1_down = run_two_group_test(
    metro, small, "metro", "small", mwu_alternative="less")

practically_significant = abs(w1["cohens_d"]) >= 0.10
verdict = "SUPPORTED" if (w1["p"] < 0.05 and w1["mean_diff"] < 0 and practically_significant) else "NOT practically supported"

print(f"\n[Data Scientist's Interpretation] Welch's t comes back p={w1['p']:.1e}, which "
      f"LOOKS like a slam dunk, but Cohen's d is only {w1['cohens_d']:+.3f} - that's about "
      f"1/50th of what's usually considered even a 'small' effect (0.2). The actual mean gap "
      f"is {w1['mean_diff']:+.1f} m^2 (117 vs 119 m^2ish), and even the t-test's own 95% CI "
      f"for that gap, [{w1['ci95'][0]:.1f}, {w1['ci95'][1]:.1f}] m^2, is basically a rounding "
      f"error - a couple of square meters either way. The MEDIANS are flat-out identical (100 "
      f"vs 100). Mann-Whitney agrees: rank-biserial is {mw1['rank_biserial']:+.3f}, also "
      f"basically nothing.\n"
      f"The downsample check is the clearest tell here: at n=10,000/group (a completely "
      f"normal sample size for this kind of test) the Welch p-value jumps to "
      f"p={w1_down['p']:.2f} - not significant anymore. So the 'significant' result at full "
      f"scale is really just n doing the work, not a real difference in home size.\n"
      f"=> H1 is {verdict}. Metro and small-city homes come out essentially the same size. "
      f"If migration pressure into big cities is real (and it probably is), it's more likely "
      f"showing up as higher PRICE per square meter rather than smaller homes - people paying "
      f"more for the same space, not accepting less space.")

## Hypothesis 2 — "Old houses were more spacious" (قدیما خونه‌ها دلبازتر بود)

**H2:** average `building_size` of old houses (`construction_year < 1396`) is bigger than newer houses. This is basically testing a piece of folk wisdom / nostalgia against the actual listings.

Same `clean_size` frame, same toolkit as H1 — Shapiro + adaptive alpha, Welch's t (mean) + Mann-Whitney (stochastic dominance), effect sizes, CIs, and the n=10,000 downsample check.

In [ ]:
# ============================================================================
# HYPOTHESIS 2 - old (<1396) vs new building_size
# ============================================================================
old = clean_size.loc[clean_size["is_old_house"] == True, "building_size"]
new = clean_size.loc[clean_size["is_old_house"] == False, "building_size"]

w2, mw2, w2_down = run_two_group_test(
    old, new, "old", "new", mwu_alternative="greater")

print(f"\n[Data Scientist's Interpretation] This one's the opposite story from H1. Welch's t: "
      f"mean_diff={w2['mean_diff']:+.1f} m^2 (old houses smaller, not bigger), "
      f"Cohen's d={w2['cohens_d']:+.3f} - that's crossing out of 'negligible' into small-but-real "
      f"territory. Mann-Whitney backs it up even more strongly: rank-biserial="
      f"{mw2['rank_biserial']:+.3f}, a moderate effect, and the one-sided p-value for "
      f"'old > new' comes back at p={mw2['p']:.2f} - i.e. there's essentially zero evidence "
      f"FOR the nostalgia and a lot of evidence against it. Median gap is about "
      f"{old.median()-new.median():+.0f} m^2 (old={old.median():.0f} vs new={new.median():.0f}), "
      f"and the t-test's 95% CI on the mean gap, [{w2['ci95'][0]:.1f}, {w2['ci95'][1]:.1f}] m^2, "
      f"sits entirely on the 'new is bigger' side - nowhere near zero.\n"
      f"Unlike H1, this one SURVIVES the downsample check (p={w2_down['p']:.1e} even at "
      f"n=10,000/group, d={w2_down['cohens_d']:+.3f}) - so this isn't just massive-n noise, it's "
      f"a real pattern in the data.\n"
      f"=> H2 is REJECTED - actually reversed. New homes measure bigger, not older ones. My "
      f"best guess at the mechanism: the 'قدیما دلبازتر بود' feeling probably comes from "
      f"courtyard houses / traditional villas, which have mostly aged out of the CURRENT "
      f"for-sale/for-rent listings pool (they're inherited, torn down, or off-market), while "
      f"what's actually still being listed as 'pre-1396' is smaller, older inner-city "
      f"apartment stock. New construction, by contrast, gets built and marketed specifically "
      f"on square footage. So the nostalgia might be true about houses that no longer show up "
      f"in this dataset - it just isn't true about what's actually for sale today.")